# UltraLight VM-UNet — ISIC2017 replication

Reproduces Table 1 of Wu et al., *Patterns* 6, 101298 (2025).

**Target:** DSC 0.9091 · IoU 0.8334 · ACC 0.9646 · SE 0.9053 · SP 0.9790 · Prec 0.9481
at 0.049 M params / 0.060 GFLOPs.

### Before running

| setting | value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Internet | **on** |
| Add-ons → Secrets | **`HF_TOKEN`** (the dataset repo is private) |

Run the cells in order. Each one fails loudly rather than silently continuing.

## 1. What hardware did we get

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

import torch
print()
print('torch      :', torch.__version__)
print('cuda       :', torch.version.cuda, '| available:', torch.cuda.is_available())
print('devices    :', torch.cuda.device_count())

## 2. Bootstrap

`check=True` on every step, so a failed clone raises here instead of leaving later
cells running in the wrong directory.

In [ ]:
import os, subprocess, sys

REPO = "https://github.com/BRUH-MAIN/UltralightUNET.git"
DEST = "/kaggle/working/repo"

subprocess.run(["rm", "-rf", DEST], check=True)
subprocess.run(["git", "clone", "--depth", "1", REPO, DEST], check=True)
os.chdir(DEST)
sys.path.insert(0, DEST)

# Kaggle images already carry torch, timm, einops, scipy, sklearn, matplotlib, tqdm.
subprocess.run(["pip", "install", "-q", "thop", "huggingface_hub", "pytest"], check=True)

print('cwd     :', os.getcwd())
print('commit  :', subprocess.run(['git','log','-1','--format=%h %s'],
                                  capture_output=True, text=True).stdout.strip())

## 3. Data

Pulls the six prepared `.npy` splits (524 MB). Preprocessing and the train/val/test
split were done once locally with `SPLIT_SEED = 42` — this notebook never re-derives
them, so the bytes here are identical to what was debugged on the dev machine.

In [ ]:
import os, subprocess

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception as e:
    print('no HF_TOKEN secret (fine only if the dataset repo is public):', e)

subprocess.run(["python", "scripts/hf_data.py", "pull",
                "--dataset", "ISIC2017", "--repo", "RohanRamesh/ultralight-vmunet-data"], check=True)

import numpy as np
for s in ('train', 'val', 'test'):
    d = np.load(f'data/ISIC2017/data_{s}.npy', mmap_mode='r')
    m = np.load(f'data/ISIC2017/mask_{s}.npy', mmap_mode='r')
    fg = (np.asarray(m[:200]) >= 128).mean() * 100
    print(f'{s:5s} {str(d.shape):24s} mask fg {fg:4.1f}%')

## 4. Sanity checks — do not skip

Cheap, and they catch the failures that would otherwise only surface hours in:

- the pure-PyTorch selective scan still matching the reference implementation on
  *this* torch build (Kaggle ships a newer one than the dev machine)
- the batched PVM layer still matching upstream's four-call form
- the parameter count still being exactly **49,457**

Expect **19 passed**. If anything fails, stop — do not train.

In [ ]:
!python -m pytest tests/ -q

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import torch
from models.UltraLight_VM_UNet import UltraLight_VM_UNet, MAMBA_BACKEND

m = UltraLight_VM_UNet().cuda()
total = sum(p.numel() for p in m.parameters())
print('backend :', MAMBA_BACKEND)
print('params  :', total, '  paper: 49457')
assert total == 49457, f'parameter count drifted: {total} != 49457'
print('OK — structural match to the paper')
del m; torch.cuda.empty_cache()

## 5. (optional) Where does throughput peak on this GPU

The model is 0.049 M parameters, so VRAM is never the limit — a batch of 8 uses under
1 GB of 15. The cost is kernel-launch overhead: the selective scan issues ~176
sequential launches per forward *regardless of batch size*, so a larger batch amortises
a fixed cost over more images.

Useful for planning your own experiments later. **Do not change `config.batch_size` for
the replication run** — 8 is the paper's hyperparameter, and 32 would mean 39 optimiser
steps per epoch instead of 157, which changes the result.

In [ ]:
!python scripts/bench_batch.py

## 6. Train

250 epochs, then automatic evaluation of the best checkpoint on the test split.
Run 1 took **1.64 h** on a single T4 (~23 s/epoch).

Watch the first epoch's timing against that — this workload is bound by CPU launch
overhead rather than GPU throughput, so it is sensitive to what CPU the session got.

In [ ]:
!python train.py

## 7. Results

`/kaggle/working` is wiped when the session ends. Download `results.zip` before then.

In [ ]:
!tail -3 results/*/log/train.info.log
print()
!du -sh results/*
!cd /kaggle/working/repo && zip -qr /kaggle/working/results.zip results/
!ls -lh /kaggle/working/results.zip

---

## Using both T4s

Run two independent experiments concurrently rather than splitting one across both.
`DataParallel` re-replicates the module every step and runs replicas in GIL-contending
threads — it adds to exactly the launch overhead that dominates here, and it would halve
the per-GPU batch from the paper's 8 to 4. One run per GPU also matches the paper's
single-V100 setup exactly.

```python
import os, subprocess
envs = [{**os.environ, 'CUDA_VISIBLE_DEVICES': str(i)} for i in (0, 1)]
p0 = subprocess.Popen(['python', 'train.py'], env=envs[0],
                      stdout=open('/kaggle/working/run0.log', 'w'),
                      stderr=subprocess.STDOUT)
# edit configs/config_setting.py:datasets before launching the second
```

## Session limits

Kaggle caps a GPU session at 12 h and 30 GPU-hours/week. `train.py` writes
`checkpoints/latest.pth` every epoch and resumes from it automatically, so a run that
outlives a session can continue if you restore the previous `results/<run>/` directory
first — keep it in a Kaggle Dataset between sessions.

## If something breaks

Kaggle ships newer torch/timm/numpy than the pinned local versions:

- `from timm.models.layers import trunc_normal_` is a deprecated shim in timm 1.x. If it
  errors, change it to `from timm.layers import trunc_normal_` in
  `models/UltraLight_VM_UNet.py`.
- torch ≥ 2.6 defaults `torch.load(weights_only=True)`. These checkpoints hold only
  tensors and plain numbers so they load fine, but a `_pickle.UnpicklingError` on resume
  would mean passing `weights_only=False` in `train.py`.

Cell 4 catches both before any training time is spent.